## Diff vs CRPS — multi-dataset comparison (2026-03-16 collated)

This notebook compares AutoCast evaluation outputs for runs whose `run_name` starts with `diff_` (diffusion) versus `crps_` (CRPS), using the collated results directory:

- `../outputs/2026-03-16_collated/`

You can select any subset of dataset modules from a single config cell and choose the per-timestep channel file (`channel_0`, `channel_1`, ..., or `channel_all`).

**Metrics covered**

- Accuracy: VRMSE (overall + rollout windows)
- Coverage: `coverage` (overall + rollout windows) + calibration curves
- Efficiency/complexity: training epoch time, inference latency/throughput, params, GFLOPs
- Lead-time behavior: metrics vs timestep from `rollout_metrics_per_timestep_channel_*.csv`


In [ ]:
from __future__ import annotations

import math
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Config: collated results root.
# You can set an absolute path, or a relative path from your current kernel cwd.
OUTPUTS_DIR = "outputs/2026-03-16_collated"


def resolve_results_root(outputs_dir: str) -> Path:  # noqa: D103
    candidate_strings = [
        outputs_dir,
        "outputs/2026-03-16_collated",
        "../outputs/2026-03-16_collated",
    ]

    seen = set()
    for c in candidate_strings:
        if c in seen:
            continue
        seen.add(c)
        p = Path(c).expanduser()
        p = (Path.cwd() / p).resolve() if not p.is_absolute() else p.resolve()
        if p.exists() and p.is_dir():
            return p

    # Fallback: search upward for repo-like root containing outputs.
    for parent in [Path.cwd(), *Path.cwd().parents]:
        p = (parent / "outputs" / "2026-03-16_collated").resolve()
        if p.exists() and p.is_dir():
            return p

    # Last resort keeps diagnostics explicit.
    p = Path(outputs_dir).expanduser()
    return (Path.cwd() / p).resolve() if not p.is_absolute() else p.resolve()


RESULTS_ROOT = resolve_results_root(OUTPUTS_DIR)

# Config: dataset modules to include.
# - None or [] => use all discovered datasets.
# - Otherwise provide module names, e.g. ["ad64", "cns64", "sw2d64"].
# SELECTED_DATASETS: list[str] | None = None
SELECTED_DATASETS: list[str] | None = [
    "rd64",
    "gs64",
    "ad64",
    "sw2d64",
    "lb128x32",
    "cns64",
    "gpe_ri_high_complexity",
]

# Config: how to decide "small" vs "large".
# - "params_model_total": includes encoder+processor+decoder
# - "params_processor_total": processor-only (often the more apples-to-apples comparison)
MODEL_SCALE_PARAM_COL = "params_model_total"

# Config: per-timestep metrics channel.
# - int (0,1,2,...) -> rollout_metrics_per_timestep_channel_<int>.csv
# - "all" -> rollout_metrics_per_timestep_channel_all.csv
CHANNEL_SELECTION: int | str = 0
METRICS_PER_TIMESTEP_FILE_TEMPLATE = "rollout_metrics_per_timestep_channel_{channel}.csv"

# Optional friendly labels for module names.
DATASET_LABEL_OVERRIDES = {
    "ad64": "AD64",
    "adm32": "ADM32",
    "cns64": "CNS64",
    "gpe_ri_high_complexity": "GPE-RI high",
    "gpe_ri_low_complexity": "GPE-RI low",
    "gpehc64": "GPEHC64",
    "gpelc64": "GPELC64",
    "gs64": "GS64",
    "lb128x32": "LB128x32",
    "rd64": "RD64",
    "sw2d464": "SW2D464",
    "sw2d64": "SW2D64",
}

ROLL_WINDOWS = ["0-1", "0-4", "6-12", "13-30", "31-99"]


def dataset_label_from_module(dataset_module: str) -> str:
    if dataset_module in DATASET_LABEL_OVERRIDES:
        return DATASET_LABEL_OVERRIDES[dataset_module]
    return dataset_module.replace("_", " ").title()


def normalize_channel_token(channel_selection: int | str) -> str:
    if isinstance(channel_selection, str) and channel_selection.lower() == "all":
        return "all"
    if isinstance(channel_selection, str) and channel_selection.isdigit():
        return channel_selection
    if isinstance(channel_selection, int):
        return str(channel_selection)
    raise ValueError(f"Unsupported CHANNEL_SELECTION={channel_selection!r}; use int or 'all'.")


CHANNEL_TOKEN = normalize_channel_token(CHANNEL_SELECTION)
PER_TIMESTEP_FILENAME = METRICS_PER_TIMESTEP_FILE_TEMPLATE.format(channel=CHANNEL_TOKEN)

if not RESULTS_ROOT.exists():
    print(f"Warning: RESULTS_ROOT does not exist: {RESULTS_ROOT}")

pd.set_option("display.max_columns", 300)
plt.rcParams.update({"figure.dpi": 120})


In [ ]:
# Direct loader for collated outputs.
# This avoids RunCollator edge-cases with symlinked collated directories.

def parse_dataset_from_run_name(run_name: str) -> str | None:
    m = re.match(
        r"^(?:diff|crps)_(?P<dataset>[a-z0-9_]+?)_(?:fno_concat|flow_matching_vit)_[0-9a-f]{7}_[0-9a-f]{7}$",
        run_name,
    )
    return m.group("dataset") if m else None


def load_single_run_metrics(run_dir: Path) -> dict:
    row: dict[str, object] = {
        "run_name": run_dir.name,
        "run_path": run_dir.name,
        "dataset": parse_dataset_from_run_name(run_dir.name),
    }

    eval_dir = run_dir / "eval"

    # evaluation_metrics.csv -> overall_* metrics
    p_eval = eval_dir / "evaluation_metrics.csv"
    if p_eval.exists():
        em = pd.read_csv(p_eval)
        if not em.empty and "window" in em.columns:
            em = em.copy()
            em["window"] = em["window"].astype(str)
            if "batch_idx" in em.columns:
                em["batch_idx"] = em["batch_idx"].astype(str)
                overall = em[(em["window"] == "all") & (em["batch_idx"] == "all")]
                if overall.empty:
                    overall = em[em["window"] == "all"]
            else:
                overall = em[em["window"] == "all"]

            if not overall.empty:
                rr = overall.iloc[0]
                for metric in ["mse", "mae", "rmse", "vrmse", "coverage"]:
                    if metric in overall.columns:
                        row[f"overall_{metric}"] = pd.to_numeric(rr[metric], errors="coerce")

    # rollout_metrics.csv -> metric_<window> columns (e.g. vrmse_0-4)
    p_roll = eval_dir / "rollout_metrics.csv"
    if p_roll.exists():
        rm = pd.read_csv(p_roll)
        if not rm.empty and "window" in rm.columns:
            rm = rm.copy()
            rm["window"] = rm["window"].astype(str)
            if "batch_idx" in rm.columns:
                rm = rm[rm["batch_idx"].astype(str) == "all"]
            for _, rr in rm.iterrows():
                w = rr["window"]
                for metric in ["mse", "mae", "rmse", "vrmse", "coverage"]:
                    if metric in rm.columns:
                        row[f"{metric}_{w}"] = pd.to_numeric(rr[metric], errors="coerce")

    # benchmark_metrics.csv -> model_* and rollout_* columns
    p_bench = eval_dir / "benchmark_metrics.csv"
    if p_bench.exists():
        bm = pd.read_csv(p_bench)
        if not bm.empty and {"benchmark_type", "metric", "value"}.issubset(bm.columns):
            for _, rr in bm.iterrows():
                prefix = str(rr["benchmark_type"])
                metric = str(rr["metric"])
                row[f"{prefix}_{metric}"] = pd.to_numeric(rr["value"], errors="coerce")

    # evaluation_metadata.csv -> train_* and params_*
    p_meta = eval_dir / "evaluation_metadata.csv"
    if p_meta.exists():
        md = pd.read_csv(p_meta)
        if not md.empty and {"category", "metric", "value"}.issubset(md.columns):
            md = md.copy()
            md["category"] = md["category"].astype(str)
            md["metric"] = md["metric"].astype(str)

            rt = md[md["category"] == "runtime_train"]
            for metric_name, out_col in [
                ("mean_epoch_s", "train_mean_epoch_s"),
                ("total_s", "train_total_s"),
                ("min_epoch_s", "train_min_epoch_s"),
                ("max_epoch_s", "train_max_epoch_s"),
            ]:
                s = rt.loc[rt["metric"] == metric_name, "value"]
                if not s.empty:
                    row[out_col] = pd.to_numeric(s.iloc[0], errors="coerce")

            params = md[md["category"] == "params"]
            for metric_name, out_col in [
                ("model_total", "params_model_total"),
                ("model_trainable", "params_model_trainable"),
                ("processor_total", "params_processor_total"),
                ("encoder_total", "params_encoder_total"),
                ("decoder_total", "params_decoder_total"),
            ]:
                s = params.loc[params["metric"] == metric_name, "value"]
                if not s.empty:
                    row[out_col] = pd.to_numeric(s.iloc[0], errors="coerce")

    return row


run_dirs = sorted(
    [p for p in RESULTS_ROOT.iterdir() if p.is_dir() and (p / "eval" / "evaluation_metrics.csv").exists()],
    key=lambda p: p.name,
)

rows = [load_single_run_metrics(run_dir) for run_dir in run_dirs]
df_metrics = pd.DataFrame(rows)

# Keep object for downstream compatibility (fallbacks check this name).
metadata_by_run = {r["run_path"]: {} for r in rows if r.get("run_path")}

print(f"Discovered run dirs: {len(run_dirs)}")
print(f"Loaded metrics rows: {len(df_metrics)}")
print(f"Metadata runs: {len(metadata_by_run)}")
print(f"Results root: {RESULTS_ROOT}")
print(f"Results root exists: {RESULTS_ROOT.exists()}")

if df_metrics.empty:
    print("Warning: no metrics rows loaded. Check RESULTS_ROOT and eval CSV availability.")

df_metrics.head()

In [ ]:
def parse_run_name(run_name: str | None) -> tuple[str | None, str | None]:
    if not run_name:
        return None, None

    s = str(run_name)

    # Primary parser for this collated naming scheme.
    m = re.match(
        r"^(diff|crps)_(?P<dataset>[a-z0-9_]+?)_(?:fno_concat|flow_matching_vit)_[0-9a-f]{7}_[0-9a-f]{7}$",
        s,
    )
    if m:
        return m.group(1), m.group("dataset")

    # Fallback parser: keep family and try to strip known model suffixes.
    fam = None
    m_fam = re.match(r"^(diff|crps)_", s)
    if m_fam:
        fam = m_fam.group(1)

    m_body = re.match(r"^(?:diff|crps)_(?P<body>.+)_[0-9a-f]{7}_[0-9a-f]{7}$", s)
    if not m_body:
        return fam, None

    body = m_body.group("body")
    for model_suffix in ["flow_matching_vit", "fno_concat"]:
        suffix = f"_{model_suffix}"
        if body.endswith(suffix):
            return fam, body[: -len(suffix)]

    return fam, None


def normalize_dataset_module(dataset: str | None) -> str | None:
    if dataset is None or (isinstance(dataset, float) and pd.isna(dataset)):
        return None
    s = str(dataset)
    m = re.match(r"^(?P<base>.+)_(?P<suffix>[0-9a-f]{6,}|\d{6,})$", s)
    return m.group("base") if m else s


df = df_metrics.copy()
df["dataset_raw"] = df.get("dataset")

if df.empty and len(df.columns) == 0:
    raise ValueError(
        "RunCollator returned an empty metrics table (no columns). "
        f"Check RESULTS_ROOT and available runs under: {RESULTS_ROOT}"
    )

# Some collations may not expose `run_name` directly.
run_name_col = next((c for c in ["run_name", "name", "run", "run_id"] if c in df.columns), None)
if run_name_col is None and "run_path" in df.columns:
    df["run_name"] = df["run_path"].astype(str).map(lambda s: Path(s).name)
elif run_name_col is not None and run_name_col != "run_name":
    df["run_name"] = df[run_name_col].astype(str)

# If we still don't have run_name, fall back to metadata_by_run.
if "run_name" not in df.columns:
    mb = globals().get("metadata_by_run")
    if isinstance(mb, dict) and mb:
        try:
            from autocast.scripts.utils import flatten_metadata_by_run

            md_wide_fallback = flatten_metadata_by_run(mb)
            if "run_path" in md_wide_fallback.columns:
                df = md_wide_fallback.copy()
                df["dataset_raw"] = df.get("dataset", df.get("data_module", pd.NA))
                if "run_name" not in df.columns:
                    df["run_name"] = df["run_path"].astype(str).map(lambda s: Path(s).name)
            else:
                msg = "flatten_metadata_by_run output missing 'run_path' column"
                raise ValueError(msg)
        except Exception as e:
            msg = (
                "Could not infer run identifiers from collated outputs or "
                f" metadata_by_run. Available df columns: "
                f"{sorted(df.columns.tolist())}. "
                f"Fallback error: {e}"
            )
            raise ValueError(msg) from e

if "run_name" not in df.columns:
    raise ValueError(
        "Missing run identifier column even after metadata_by_run fallback. "
        f"Available columns: {sorted(df.columns.tolist())}"
    )

# Ensure run_path exists for downstream merges and CSV loading.
if "run_path" not in df.columns:
    run_path_from_name = None
    mb = globals().get("metadata_by_run")
    if isinstance(mb, dict) and mb:
        by_name = {Path(str(k)).name: str(k) for k in mb.keys()}
        run_path_from_name = df["run_name"].astype(str).map(by_name)

    if run_path_from_name is not None:
        df["run_path"] = run_path_from_name.where(run_path_from_name.notna(), df["run_name"].astype(str))
    else:
        df["run_path"] = df["run_name"].astype(str)

parsed = df["run_name"].map(parse_run_name)
df["loss_family"] = parsed.map(lambda x: x[0] if isinstance(x, tuple) else None)
df["dataset_module"] = parsed.map(lambda x: x[1] if isinstance(x, tuple) else None)

# Fallback to dataset column for any run where dataset module couldn't be parsed from run_name.
fallback_dataset = df["dataset_raw"].map(normalize_dataset_module)
df["dataset_module"] = df["dataset_module"].where(df["dataset_module"].notna(), fallback_dataset)

df = df[df["loss_family"].isin(["diff", "crps"])].copy()

discovered_datasets = sorted(df["dataset_module"].dropna().astype(str).unique().tolist())
if not discovered_datasets:
    raise ValueError("No dataset modules discovered from collated outputs.")

if not SELECTED_DATASETS:
    selected_datasets = discovered_datasets
else:
    selected_datasets = [d for d in SELECTED_DATASETS if d in discovered_datasets]
    missing_requested = sorted(set(SELECTED_DATASETS) - set(selected_datasets))
    if missing_requested:
        print(f"Warning: requested datasets not discovered: {missing_requested}")

if not selected_datasets:
    raise ValueError("No datasets selected after filtering.")

df = df[df["dataset_module"].isin(selected_datasets)].copy()

df["dataset_family"] = df["dataset_module"]
df["dataset_label"] = df["dataset_module"].map(dataset_label_from_module)


def assign_model_scale(df_in: pd.DataFrame) -> pd.Series:
    param_col = globals().get("MODEL_SCALE_PARAM_COL", "params_model_total")
    if param_col not in df_in.columns:
        print(f"Warning: {param_col} not in df; falling back to params_model_total")
        param_col = "params_model_total"

    if param_col not in df_in.columns:
        return pd.Series("large", index=df_in.index)

    out = pd.Series("large", index=df_in.index, dtype="object")
    params = pd.to_numeric(df_in[param_col], errors="coerce")

    # Classify small/large per (dataset_module, loss_family), not globally.
    group_cols = [c for c in ["dataset_module", "loss_family"] if c in df_in.columns]
    if len(group_cols) < 2:
        return out

    for _, g in df_in.groupby(group_cols, dropna=False):
        idx = g.index
        p = params.loc[idx]
        vals = sorted(p.dropna().unique().tolist())
        if len(vals) <= 1:
            out.loc[idx] = "large"
            continue

        low, high = vals[0], vals[-1]
        out.loc[p[p == low].index] = "small"
        out.loc[p[p == high].index] = "large"

        # If there are more than two parameter values in a group,
        # map middle values to nearest endpoint so we keep a two-variant palette.
        mid_idx = p[(p != low) & (p != high)].index
        for ii in mid_idx:
            val = p.loc[ii]
            if pd.isna(val):
                out.loc[ii] = "large"
            else:
                out.loc[ii] = "small" if abs(val - low) <= abs(val - high) else "large"

    return out


df["model_scale"] = assign_model_scale(df)
df["family_variant"] = df["loss_family"] + "_" + df["model_scale"]

# Dynamic allowlist used by downstream plotting helpers.
DATASET_ALLOWLIST = {d: dataset_label_from_module(d) for d in selected_datasets}

print(f"Discovered datasets ({len(discovered_datasets)}): {discovered_datasets}")
print(f"Selected datasets ({len(selected_datasets)}): {selected_datasets}")
print(f"Per-timestep file: {PER_TIMESTEP_FILENAME}")
print("Runs by dataset/loss:")
display(
    df.groupby(["dataset_module", "loss_family"]).size().unstack(fill_value=0).sort_index()
)
print("Runs by family variant:")
display(df.groupby(["dataset_module", "family_variant"]).size().unstack(fill_value=0).sort_index())

_cols = ["run_name", "loss_family", "dataset_raw", "dataset_module", "dataset_label"]
_cols = [c for c in _cols if c in df.columns]
_view = df[_cols].copy() if _cols else df.copy()
_sort = [c for c in ["dataset_module", "loss_family", "run_name"] if c in _view.columns]
if _sort:
    _view = _view.sort_values(_sort)
_view.head(40)

In [ ]:
# Metadata columns are already loaded directly in cell 2 from eval CSVs.
# Keep this cell as a light diagnostics check.
cols_meta = sorted(
    [c for c in df.columns if any(k in c for k in ["train_", "params_", "model_", "rollout_"])]
)

print(f"Metadata/perf columns found: {len(cols_meta)}")
cols_meta[:60], df.shape

## Tables

The cells below build:

- a **per-run table** for the selected dataset modules
- a **grouped summary** (mean/std/count) by `(dataset_module, loss_family)`
- **diff - crps deltas** per dataset module


In [ ]:
def existing_cols(cols: list[str]) -> list[str]:
    return [c for c in cols if c in df.columns]

metric_cols = [
    "overall_vrmse",
    "overall_coverage",
] + [f"vrmse_{w}" for w in ROLL_WINDOWS] + [f"coverage_{w}" for w in ROLL_WINDOWS]

perf_cols = existing_cols(metric_cols)

speed_cols = existing_cols(
    [
        "train_mean_epoch_s",
        "train_total_s",
        "params_model_total",
        "params_model_trainable",
        "model_latency_ms_per_sample",
        "model_latency_ms_per_batch",
        "model_throughput_samples_per_sec",
        "model_gflops_per_sample",
        "model_peak_gpu_memory_mb",
        "rollout_latency_ms_per_step",
        "rollout_latency_ms_per_sample",
        "rollout_throughput_samples_per_sec",
        "rollout_peak_gpu_memory_mb",
    ]
)

id_cols = existing_cols(
    [
        "run_name",
        "run_path",
        "dataset_label",
        "dataset_family",
        "loss_family",
        "processor_model",
        "processor_hidden_channels",
    ]
)

per_run = df[id_cols + perf_cols + speed_cols].copy()

sort_cols = [c for c in ["dataset_family", "loss_family", "run_name"] if c in per_run.columns]
if sort_cols:
    per_run = per_run.sort_values(sort_cols)

per_run

In [ ]:
group_keys = ["dataset_family", "dataset_label", "loss_family"]

summary_cols = existing_cols(
    [
        "overall_vrmse",
        "overall_coverage",
        "train_mean_epoch_s",
        "model_latency_ms_per_sample",
        "model_throughput_samples_per_sec",
        "rollout_latency_ms_per_step",
        "rollout_throughput_samples_per_sec",
        "params_model_total",
        "model_gflops_per_sample",
    ]
    + [f"vrmse_{w}" for w in ROLL_WINDOWS]
    + [f"coverage_{w}" for w in ROLL_WINDOWS]
)

summary = (
    df.groupby(group_keys)[summary_cols]
    .agg(["mean", "std", "count"])
    .sort_index()
)
summary

In [ ]:
# Diff - CRPS deltas (per dataset) for primary metrics.
primary = existing_cols(
    [
        "overall_vrmse",
        "overall_coverage",
        "train_mean_epoch_s",
        "model_latency_ms_per_sample",
        "model_throughput_samples_per_sec",
        "params_model_total",
        "model_gflops_per_sample",
    ]
)

means = df.groupby(["dataset_family", "loss_family"])[primary].mean(numeric_only=True)

# Wide: columns like ('overall_vrmse','diff')
wide = means.unstack("loss_family")

col_families = set(wide.columns.get_level_values(1)) if isinstance(wide.columns, pd.MultiIndex) else set()

if {"diff", "crps"}.issubset(col_families):
    try:
        deltas = wide.xs("diff", axis=1, level=1) - wide.xs("crps", axis=1, level=1)
        deltas = deltas.rename(columns={c: f"diff_minus_crps__{c}" for c in deltas.columns})
    except KeyError as e:
        print(f"Could not compute diff-crps deltas: {e}")
        deltas = pd.DataFrame(index=wide.index)
else:
    print(f"Missing diff/crps columns in wide means: {sorted(col_families)}")
    deltas = pd.DataFrame(index=wide.index)

# Attach friendly dataset labels.
deltas = deltas.reset_index().merge(
    df[["dataset_family", "dataset_label"]].drop_duplicates(), on="dataset_family", how="left"
)
deltas = deltas.set_index(["dataset_family", "dataset_label"]).sort_index()

deltas

## Plots

All plots below compare **diff vs crps** within each dataset.


In [ ]:
def dataset_labels_in_order(df_in: pd.DataFrame) -> list[str]:
    if "dataset_module" in df_in.columns:
        order_df = (
            df_in[["dataset_module", "dataset_label"]]
            .dropna()
            .drop_duplicates()
        )
        label_by_module = dict(zip(order_df["dataset_module"], order_df["dataset_label"]))

        # Respect user-configured dataset order when provided.
        selected = globals().get("selected_datasets")
        if selected is None:
            selected = globals().get("SELECTED_DATASETS")

        if selected:
            ordered_modules = [m for m in selected if m in label_by_module]
            extras = sorted([m for m in label_by_module.keys() if m not in set(ordered_modules)])
            ordered_modules.extend(extras)
            return [label_by_module[m] for m in ordered_modules]

        return [label_by_module[m] for m in sorted(label_by_module.keys())]

    return list(df_in["dataset_label"].dropna().drop_duplicates())


def x_label_rotation(n_labels: int) -> int:
    if n_labels <= 4:
        return 0
    if n_labels <= 8:
        return 20
    return 35


def family_group_col(df_in: pd.DataFrame) -> str:
    return "family_variant" if "family_variant" in df_in.columns else "loss_family"


def family_groups_in_order(df_in: pd.DataFrame) -> list[str]:
    group_col = family_group_col(df_in)
    present = set(df_in[group_col].dropna().unique().tolist())
    preferred = ["crps_large", "crps_small", "crps", "diff_large", "diff_small", "diff"]
    ordered = [g for g in preferred if g in present]
    ordered.extend(sorted([g for g in present if g not in set(ordered)]))
    return ordered


FAMILY_STYLE = {
    "crps_large": {"color": "tab:blue", "label": "CRPS large"},
    "crps_small": {"color": "#9ecae1", "label": "CRPS small"},
    "crps": {"color": "tab:blue", "label": "CRPS"},
    "diff_large": {"color": "tab:orange", "label": "Diffusion large"},
    "diff_small": {"color": "#fdd0a2", "label": "Diffusion small"},
    "diff": {"color": "tab:orange", "label": "Diffusion"},
}


def grouped_bar(df_in: pd.DataFrame, metric: str, title: str, ylabel: str):
    group_col = family_group_col(df_in)
    required = ["dataset_label", group_col, metric]
    missing = [c for c in required if c not in df_in.columns]
    if missing:
        print(f"Skipping {metric}: missing columns {missing}")
        return

    d = df_in[required].dropna().copy()
    if d.empty:
        print(f"No data for {metric}")
        return

    g = d.groupby(["dataset_label", group_col], dropna=False)[metric].mean().reset_index()

    datasets = dataset_labels_in_order(df_in)
    datasets = [ds for ds in datasets if ds in set(g["dataset_label"])]
    families = [f for f in family_groups_in_order(df_in) if f in set(g[group_col])]

    x = np.arange(len(datasets), dtype=float)
    width = 0.8 / max(1, len(families))

    fig_width = max(8.0, 0.9 * len(datasets) + 2.0)
    fig, ax = plt.subplots(figsize=(fig_width, 3.8))
    for i, fam in enumerate(families):
        vals = pd.to_numeric(
            g[g[group_col] == fam]
            .set_index("dataset_label")
            .reindex(datasets)[metric],
            errors="coerce",
        ).to_numpy(dtype=float)
        offs = (i - (len(families) - 1) / 2.0) * width
        style = FAMILY_STYLE.get(fam, {"color": None, "label": fam})
        ax.bar(x + offs, vals, width=width, label=style["label"], color=style["color"])

    ax.set_xticks(list(x))
    ax.set_xticklabels(datasets, rotation=x_label_rotation(len(datasets)), ha="right")
    ax.set_ylabel(ylabel)
    ax.set_yscale("log")
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.25)
    ax.legend()
    plt.tight_layout()
    plt.show()


grouped_bar(df, "overall_vrmse", "Overall VRMSE (lower is better)", "VRMSE")


In [ ]:
if "overall_coverage" in df.columns:
    grouped_bar(df, "overall_coverage", "Overall coverage", "Coverage")

# Optional: rollout windows
for w in ROLL_WINDOWS:
    col = f"vrmse_{w}"
    if col in df.columns:
        grouped_bar(df, col, f"Rollout VRMSE window {w}", "VRMSE")
        # break  # keep notebook compact by default


In [ ]:
# Coverage vs VRMSE scatter (points = runs)
if {"overall_vrmse", "overall_coverage"}.issubset(df.columns):
    group_col = family_group_col(df)
    _cols = ["dataset_label", group_col, "overall_vrmse", "overall_coverage", "run_name"]
    _cols = [c for c in _cols if c in df.columns]
    d = df[_cols].dropna()

    dataset_labels = dataset_labels_in_order(d)
    n = len(dataset_labels)
    ncols = min(4, max(1, n))
    nrows = int(math.ceil(n / ncols))

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(4.2 * ncols, 3.2 * nrows),
        sharex=False,
        sharey=False,
    )
    axes = np.array(axes).reshape(-1)

    for i, ds_label in enumerate(dataset_labels):
        ax = axes[i]
        sub = d[d["dataset_label"] == ds_label]
        for fam in family_groups_in_order(d):
            ss = sub[sub[group_col] == fam]
            if ss.empty:
                continue
            style = FAMILY_STYLE.get(fam, {"color": None, "label": fam})
            marker = "^" if fam.startswith("diff") else "o"
            ax.scatter(
                ss["overall_vrmse"],
                ss["overall_coverage"],
                label=style["label"],
                marker=marker,
                alpha=0.85,
                color=style["color"],
            )
        ax.set_title(ds_label)
        ax.set_xlabel("overall_vrmse")
        ax.set_ylabel("overall_coverage")
        ax.grid(alpha=0.25)

    for ax in axes[n:]:
        ax.set_visible(False)

    handles, labels = axes[0].get_legend_handles_labels() if n else ([], [])
    if handles:
        fig.legend(handles, labels, loc="upper center", ncol=2)
    plt.tight_layout(rect=(0, 0, 1, 0.94))
    plt.show()
else:
    print("Missing overall_vrmse/overall_coverage")


In [ ]:
# Timing / throughput / complexity comparisons (group means)
for metric, title, ylabel in [
    ("train_mean_epoch_s", "Mean epoch time (s) (lower is better)", "seconds"),
    ("model_latency_ms_per_sample", "Model latency per sample (ms) (lower is better)", "ms"),
    ("model_throughput_samples_per_sec", "Model throughput (samples/s) (higher is better)", "samples/s"),
    ("params_model_total", "Model parameters (total)", "params"),
    ("model_gflops_per_sample", "GFLOPs per sample", "GFLOPs"),
]:
    if metric in df.columns:
        grouped_bar(df, metric, title, ylabel)


## Training time comparison

These plots focus specifically on **training time**, using `evaluation_metadata.csv` fields when available.


In [ ]:
def first_present(df_in: pd.DataFrame, candidates: list[str]) -> str | None:
    for c in candidates:
        if c in df_in.columns:
            return c
    return None


def resolve_training_columns(df_in: pd.DataFrame) -> tuple[pd.DataFrame, str | None, str | None]:
    out = df_in.copy()

    epoch_candidates = ["train_mean_epoch_s", "evaluation_runtime_train_mean_epoch_s"]
    total_candidates = ["train_total_s", "evaluation_runtime_train_total_s"]

    col_epoch = first_present(out, epoch_candidates)
    col_total = first_present(out, total_candidates)

    # Fallback 1: rebuild from metadata_by_run using flatten helper.
    if (col_epoch is None and col_total is None) and "metadata_by_run" in globals():
        try:
            from autocast.scripts.utils import flatten_metadata_by_run

            md_wide_retry = flatten_metadata_by_run(metadata_by_run)
            needed = [c for c in epoch_candidates + total_candidates if c in md_wide_retry.columns]
            if needed and "run_path" in out.columns:
                out = out.drop(columns=[c for c in needed if c in out.columns], errors="ignore")
                out = out.merge(md_wide_retry[["run_path"] + needed], on="run_path", how="left")
                col_epoch = first_present(out, epoch_candidates)
                col_total = first_present(out, total_candidates)
        except Exception as e:
            print(f"metadata_by_run fallback failed: {e}")

    # Fallback 2: read training runtime directly from evaluation_metadata.csv.
    if (col_epoch is None and col_total is None) and "run_path" in out.columns:
        rows = []
        base = RESULTS_ROOT
        for run_path in out["run_path"].dropna().astype(str).unique():
            meta_csv = base / run_path / "eval" / "evaluation_metadata.csv"
            if not meta_csv.exists():
                continue
            try:
                m = pd.read_csv(meta_csv)
                if not {"category", "metric", "value"}.issubset(m.columns):
                    continue
                mm = m[m["category"] == "runtime_train"]
                if mm.empty:
                    continue
                row = {"run_path": run_path}
                for metric_name, out_col in [
                    ("mean_epoch_s", "train_mean_epoch_s"),
                    ("total_s", "train_total_s"),
                ]:
                    s = mm.loc[mm["metric"] == metric_name, "value"]
                    if not s.empty:
                        row[out_col] = pd.to_numeric(s.iloc[0], errors="coerce")
                rows.append(row)
            except Exception:
                pass

        if rows:
            rt = pd.DataFrame(rows)
            out = out.drop(columns=[c for c in ["train_mean_epoch_s", "train_total_s"] if c in out.columns], errors="ignore")
            out = out.merge(rt, on="run_path", how="left")
            col_epoch = first_present(out, epoch_candidates)
            col_total = first_present(out, total_candidates)

    return out, col_epoch, col_total


df, col_epoch, col_total = resolve_training_columns(df)

print("epoch_time_col:", col_epoch)
print("total_train_time_col:", col_total)


def grouped_bar_with_error(df_in: pd.DataFrame, metric: str, title: str, ylabel: str):
    group_col = family_group_col(df_in)
    required = ["dataset_label", group_col, metric]
    missing = [c for c in required if c not in df_in.columns]
    if missing:
        print(f"Skipping {metric}: missing columns {missing}")
        return

    d = df_in[required].dropna().copy()
    if d.empty:
        print(f"No data for {metric}")
        return

    stats = (
        d.groupby(["dataset_label", group_col], dropna=False)[metric]
        .agg(["mean", "std", "count"])
        .reset_index()
    )

    datasets = dataset_labels_in_order(df_in)
    datasets = [ds for ds in datasets if ds in set(stats["dataset_label"])]
    families = [f for f in family_groups_in_order(df_in) if f in set(stats[group_col])]

    x = np.arange(len(datasets), dtype=float)
    width = 0.8 / max(1, len(families))

    fig_width = max(8.0, 0.9 * len(datasets) + 2.0)
    fig, ax = plt.subplots(figsize=(fig_width, 3.8))
    for i, fam in enumerate(families):
        sub = stats[stats[group_col] == fam].set_index("dataset_label").reindex(datasets)
        vals = pd.to_numeric(sub["mean"], errors="coerce").to_numpy(dtype=float)
        # If count==1, std is NaN; treat as 0 for errorbar.
        yerr = pd.to_numeric(sub["std"], errors="coerce").fillna(0.0).to_numpy(dtype=float)

        offs = (i - (len(families) - 1) / 2.0) * width
        style = FAMILY_STYLE.get(fam, {"color": None, "label": fam})
        ax.bar(x + offs, vals, width=width, label=style["label"], yerr=yerr, capsize=4, color=style["color"])

    ax.set_xticks(list(x))
    ax.set_xticklabels(datasets, rotation=x_label_rotation(len(datasets)), ha="right")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.25)
    ax.legend()
    plt.tight_layout()
    plt.show()


if col_epoch:
    grouped_bar_with_error(df, col_epoch, "Training time: mean epoch seconds (lower is better)", "seconds")

if col_total:
    grouped_bar_with_error(df, col_total, "Training time: total train seconds (lower is better)", "seconds")

# Per-run scatter for epoch time (useful even when n=1)
group_col = family_group_col(df)
if col_epoch and {"dataset_label", group_col}.issubset(df.columns):
    d = df[["dataset_label", group_col, col_epoch]].dropna()
    ds_order = dataset_labels_in_order(d)
    fig_width = max(8.0, 0.9 * len(ds_order) + 2.0)
    fig, ax = plt.subplots(figsize=(fig_width, 3.8))
    groups = [g for g in family_groups_in_order(d) if g in set(d[group_col])]
    for i, fam in enumerate(groups):
        ss = d[d[group_col] == fam]
        if ss.empty:
            continue
        # Jitter x by dataset index and family group for readability.
        x = ss["dataset_label"].map({ds: i for i, ds in enumerate(ds_order)}).astype(float)
        offset = (i - (len(groups) - 1) / 2.0) * 0.06
        style = FAMILY_STYLE.get(fam, {"color": None, "label": fam})
        marker = "^" if fam.startswith("diff") else "o"
        ax.scatter(x + offset, ss[col_epoch], label=style["label"], marker=marker, alpha=0.85, color=style["color"])

    ax.set_xticks(range(len(ds_order)))
    ax.set_xticklabels(ds_order, rotation=x_label_rotation(len(ds_order)), ha="right")
    ax.set_ylabel("mean_epoch_s")
    ax.set_title("Training time per run: mean_epoch_s")
    ax.grid(axis="y", alpha=0.25)
    ax.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# Total epochs comparison
# If explicit epoch count is not logged, estimate as total_s / mean_epoch_s.

def resolve_epoch_count_column(df_in: pd.DataFrame) -> tuple[pd.DataFrame, str | None, str]:
    out = df_in.copy()

    explicit_candidates = [
        "train_num_epochs",
        "evaluation_runtime_train_num_epochs",
        "evaluation_runtime_train_n_epochs",
    ]
    for c in explicit_candidates:
        if c in out.columns:
            return out, c, "explicit"

    col_epoch = first_present(out, ["train_mean_epoch_s", "evaluation_runtime_train_mean_epoch_s"])
    col_total = first_present(out, ["train_total_s", "evaluation_runtime_train_total_s"])

    if col_epoch and col_total:
        denom = pd.to_numeric(out[col_epoch], errors="coerce")
        numer = pd.to_numeric(out[col_total], errors="coerce")
        est = (numer / denom).replace([float("inf"), -float("inf")], pd.NA)
        out["train_num_epochs_est"] = est.round().astype("Float64")
        return out, "train_num_epochs_est", "estimated"

    return out, None, "missing"


df, col_epochs, epoch_mode = resolve_epoch_count_column(df)
print("epochs_col:", col_epochs, "(mode:", epoch_mode, ")")

if col_epochs:
    grouped_bar_with_error(
        df,
        col_epochs,
        (
            "Training total epochs (explicit)"
            if epoch_mode == "explicit"
            else "Training total epochs (estimated: total_s / mean_epoch_s)"
        ),
        "epochs",
    )

    # Per-run scatter for epochs
    group_col = family_group_col(df)
    if {"dataset_label", group_col}.issubset(df.columns):
        d = df[["dataset_label", group_col, col_epochs]].dropna()
        ds_order = dataset_labels_in_order(d)
        fig_width = max(8.0, 0.9 * len(ds_order) + 2.0)
        fig, ax = plt.subplots(figsize=(fig_width, 3.8))
        groups = [g for g in family_groups_in_order(d) if g in set(d[group_col])]
        for i, fam in enumerate(groups):
            ss = d[d[group_col] == fam]
            if ss.empty:
                continue
            x = ss["dataset_label"].map({ds: i for i, ds in enumerate(ds_order)}).astype(float)
            offset = (i - (len(groups) - 1) / 2.0) * 0.06
            style = FAMILY_STYLE.get(fam, {"color": None, "label": fam})
            marker = "^" if fam.startswith("diff") else "o"
            ax.scatter(x + offset, ss[col_epochs], label=style["label"], marker=marker, alpha=0.85, color=style["color"])

        ax.set_xticks(range(len(ds_order)))
        ax.set_xticklabels(ds_order, rotation=x_label_rotation(len(ds_order)), ha="right")
        ax.set_ylabel("epochs")
        ax.set_title("Training epochs per run")
        ax.grid(axis="y", alpha=0.25)
        ax.legend()
        plt.tight_layout()
        plt.show()
else:
    print("Could not derive epoch counts from available columns.")


In [ ]:
# Coverage calibration panel:
# rows = windows (all, 0-1, 0-4, 6-12, 13-30, 31-99)
# cols = datasets

def load_window_curve(run_path: str, window_label: str) -> pd.DataFrame | None:
    if window_label == "all":
        filename = "test_coverage_window_all.csv"
    else:
        filename = f"rollout_coverage_window_{window_label}.csv"

    p = RESULTS_ROOT / run_path / "eval" / filename
    if not p.exists():
        return None

    c = pd.read_csv(p)
    if c.empty or not {"coverage_level", "observed_mean"}.issubset(c.columns):
        return None

    c = c.copy()
    c["run_path"] = run_path
    c["window"] = window_label
    return c


def build_coverage_panel_data(df_in: pd.DataFrame, windows: list[str]) -> pd.DataFrame:
    group_col = family_group_col(df_in)
    key_cols = ["run_path", "dataset_label", "loss_family"]
    if group_col != "loss_family":
        key_cols.append(group_col)
    missing = [c for c in key_cols if c not in df_in.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    base = df_in[key_cols].dropna().drop_duplicates()
    curves = []
    for _, r in base.iterrows():
        for w in windows:
            c = load_window_curve(str(r["run_path"]), w)
            if c is None:
                continue
            c["dataset_label"] = r["dataset_label"]
            c["loss_family"] = r["loss_family"]
            if "family_variant" in base.columns:
                c["family_variant"] = r["family_variant"]
            curves.append(c)

    if not curves:
        return pd.DataFrame()
    return pd.concat(curves, ignore_index=True)


WINDOW_ROWS = ["all", "0-1", "0-4", "6-12", "13-30", "31-99"]
cur_panel = build_coverage_panel_data(df, WINDOW_ROWS)

if cur_panel.empty:
    print("No coverage-curve files found for requested windows.")
else:
    dataset_cols = [label for label in dataset_labels_in_order(df) if label in set(cur_panel["dataset_label"])]

    nrows = len(WINDOW_ROWS)
    ncols = max(1, len(dataset_cols))
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(4.0 * ncols, 2.3 * nrows),
        sharex=True,
        sharey=True,
    )

    # Normalize axes shape
    if nrows == 1 and ncols == 1:
        axes = [[axes]]
    elif nrows == 1:
        axes = [list(axes)]
    elif ncols == 1:
        axes = [[ax] for ax in axes]

    family_style = FAMILY_STYLE

    for i, w in enumerate(WINDOW_ROWS):
        for j, ds_label in enumerate(dataset_cols):
            ax = axes[i][j]
            sub = cur_panel[(cur_panel["window"] == w) & (cur_panel["dataset_label"] == ds_label)]

            # Perfect calibration line
            ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1, color="black", alpha=0.6)

            if not sub.empty:
                # Plot per-run lines faintly + family mean line bold.
                group_col = "family_variant" if "family_variant" in sub.columns else "loss_family"
                groups = [g for g in family_groups_in_order(sub) if g in set(sub[group_col])]
                for fam in groups:
                    sf = sub[sub[group_col] == fam]
                    if sf.empty:
                        continue
                    style = family_style.get(fam, {"color": None, "label": fam})

                    for _, rr in sf.groupby("run_path"):
                        ax.plot(
                            rr["coverage_level"],
                            rr["observed_mean"],
                            color=style["color"],
                            alpha=0.25,
                            linewidth=1,
                        )

                    mean_curve = (
                        sf.groupby("coverage_level", as_index=False)["observed_mean"]
                        .mean()
                        .sort_values("coverage_level")
                    )
                    ax.plot(
                        mean_curve["coverage_level"],
                        mean_curve["observed_mean"],
                        color=style["color"],
                        linewidth=2,
                    )

            if i == 0:
                ax.set_title(ds_label)
            if j == 0:
                ax.set_ylabel(f"{w}\nobserved_mean")
            if i == nrows - 1:
                ax.set_xlabel("coverage_level")
            ax.grid(alpha=0.2)

    present_groups = family_groups_in_order(cur_panel)
    legend_handles = [
        plt.Line2D([0], [0], color=family_style[g]["color"], linewidth=2, label=f"{family_style[g]['label']} (mean)")
        for g in present_groups
        if g in family_style
    ]
    legend_handles.append(
        plt.Line2D([0], [0], color="black", linestyle="--", linewidth=1, label="Perfect calibration")
    )
    fig.legend(handles=legend_handles, loc="upper center", ncol=max(3, len(legend_handles)))
    plt.suptitle("Coverage calibration panel: windows x datasets", y=0.995)
    plt.tight_layout(rect=(0, 0, 1, 0.985))
    plt.show()


## Optional: coverage curves (calibration)

These plots read the per-run curve CSVs (`test_coverage_window_all.csv` and `rollout_coverage_window_*.csv`).


In [ ]:
def load_curve_csv(run_path: str, filename: str) -> pd.DataFrame | None:
    p = RESULTS_ROOT / run_path / "eval" / filename
    if not p.exists():
        return None
    c = pd.read_csv(p)
    c = c.copy()
    c["run_path"] = run_path
    return c


def plot_coverage_curves(df_in: pd.DataFrame, curve_filename: str, title: str, max_runs_per_group: int = 6):
    # Sample a few runs per dataset/family to keep plots readable.
    group_col = family_group_col(df_in)
    d = df_in[["run_path", "dataset_label", "loss_family", group_col]].drop_duplicates().copy()

    sampled = (
        d.groupby(["dataset_label", group_col], dropna=False)
        .head(max_runs_per_group)
        .reset_index(drop=True)
    )

    curves = []
    for _, r in sampled.iterrows():
        c = load_curve_csv(r["run_path"], curve_filename)
        if c is None or c.empty:
            continue
        c["dataset_label"] = r["dataset_label"]
        c["loss_family"] = r["loss_family"]
        c[group_col] = r[group_col]
        curves.append(c)

    if not curves:
        print(f"No curves found for {curve_filename}")
        return

    cur = pd.concat(curves, ignore_index=True)

    dataset_labels = [label for label in dataset_labels_in_order(df_in) if label in set(cur["dataset_label"])]
    n = len(dataset_labels)
    ncols = min(4, max(1, n))
    nrows = int(math.ceil(n / ncols))

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(4.0 * ncols, 3.0 * nrows),
        sharex=True,
        sharey=True,
    )
    axes = np.array(axes).reshape(-1)

    # Consistent style per family + figure-level legend.
    family_style = FAMILY_STYLE

    for i, ds_label in enumerate(dataset_labels):
        ax = axes[i]
        sub = cur[cur["dataset_label"] == ds_label]
        if sub.empty:
            ax.set_title(ds_label)
            continue

        # Perfect calibration line.
        ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1, alpha=0.6, color="black")

        groups = [g for g in family_groups_in_order(cur) if g in set(sub[group_col])]
        for fam in groups:
            ss = sub[sub[group_col] == fam]
            if ss.empty:
                continue
            style = family_style.get(fam, {"color": None, "label": fam})
            for run_path, sss in ss.groupby("run_path"):
                if {"coverage_level", "observed_mean"}.issubset(sss.columns):
                    ax.plot(
                        sss["coverage_level"],
                        sss["observed_mean"],
                        alpha=0.6,
                        color=style["color"],
                    )

        ax.set_title(ds_label)
        ax.set_xlabel("coverage_level")
        ax.set_ylabel("observed_mean")
        ax.grid(alpha=0.25)

    for ax in axes[n:]:
        ax.set_visible(False)

    # Family-level legend at figure scope.
    legend_handles = []
    for fam in family_groups_in_order(cur):
        if fam in set(cur[group_col].dropna().unique()):
            style = family_style.get(fam, {"color": None, "label": fam})
            handle = plt.Line2D([0], [0], color=style["color"], linewidth=2, label=style["label"])
            legend_handles.append(handle)
    legend_handles.append(
        plt.Line2D([0], [0], color="black", linestyle="--", linewidth=1, label="Perfect calibration")
    )
    fig.legend(handles=legend_handles, loc="upper center", ncol=max(2, len(legend_handles)))

    plt.suptitle(title)
    plt.tight_layout(rect=(0, 0, 1, 0.94))
    plt.show()


plot_coverage_curves(
    df,
    curve_filename="test_coverage_window_all.csv",
    title="Test coverage calibration curves (window=all)",
)

In [ ]:
# Example rollout window curve (adjust filename as needed)
plot_coverage_curves(
    df,
    # curve_filename="rollout_coverage_window_0-4.csv",
    curve_filename="rollout_coverage_window_6-12.csv",
    title="Rollout coverage calibration curves (window 6-12)",
)

In [ ]:
# Lead-time metrics from per-timestep rollout files
#
# This section loads eval/{PER_TIMESTEP_FILENAME} for each selected run
# and compares Diff vs CRPS as a function of lead time.
#
# Change CHANNEL_SELECTION in the config cell to switch between
# channel_0, channel_1, ... and channel_all.
# Missing per-timestep files are skipped and reported.


In [ ]:
def load_per_timestep_metrics(df_in: pd.DataFrame, filename: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    key_cols = ["run_path", "dataset_module", "dataset_label", "loss_family"]
    if "family_variant" in df_in.columns:
        key_cols.append("family_variant")
    base = df_in[key_cols].dropna().drop_duplicates().copy()

    rows = []
    missing_rows = []

    for r in base.itertuples(index=False):
        run_path = str(r.run_path)
        p = RESULTS_ROOT / run_path / "eval" / filename
        if not p.exists():
            missing_rows.append(
                {
                    "run_path": run_path,
                    "dataset_module": r.dataset_module,
                    "dataset_label": r.dataset_label,
                    "loss_family": r.loss_family,
                }
            )
            continue

        raw = pd.read_csv(p, index_col=0)
        if raw.empty:
            continue

        long = raw.reset_index().rename(columns={raw.index.name or "index": "metric"})
        long = long.melt(id_vars="metric", var_name="timestep", value_name="value")
        long["timestep"] = pd.to_numeric(long["timestep"], errors="coerce")
        long["value"] = pd.to_numeric(long["value"], errors="coerce")
        long = long.dropna(subset=["metric", "timestep", "value"]).copy()

        if long.empty:
            continue

        long["run_path"] = run_path
        long["dataset_module"] = r.dataset_module
        long["dataset_label"] = r.dataset_label
        long["loss_family"] = r.loss_family
        if "family_variant" in base.columns:
            long["family_variant"] = r.family_variant
        rows.append(long)

    metrics_long = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    missing_df = pd.DataFrame(missing_rows)
    return metrics_long, missing_df


def plot_metric_vs_lead_time(
    metrics_long: pd.DataFrame,
    metric: str,
    dataset_labels: list[str],
    max_timestep: int | None = None,
):
    sub = metrics_long[metrics_long["metric"] == metric].copy()
    if max_timestep is not None:
        sub = sub[sub["timestep"] <= max_timestep].copy()
    if sub.empty:
        print(f"No rows for metric={metric!r}")
        return

    labels = [d for d in dataset_labels if d in set(sub["dataset_label"])]
    n = len(labels)
    ncols = min(4, max(1, n))
    nrows = int(math.ceil(n / ncols))

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(4.2 * ncols, 3.0 * nrows),
        sharex=True,
        # sharey=True,
        sharey=False,
    )
    axes = np.array(axes).reshape(-1)

    group_col = "family_variant" if "family_variant" in sub.columns else "loss_family"
    styles = FAMILY_STYLE

    for i, ds_label in enumerate(labels):
        ax = axes[i]
        ds = sub[sub["dataset_label"] == ds_label]

        groups = [g for g in family_groups_in_order(ds) if g in set(ds[group_col])]
        for fam in groups:
            sf = ds[ds[group_col] == fam]
            if sf.empty:
                continue

            agg = (
                sf.groupby("timestep", as_index=False)["value"]
                .agg(["mean", "std", "count"])
                .reset_index()
                .sort_values("timestep")
            )

            style = styles.get(fam, {"color": None, "label": fam})
            ax.plot(agg["timestep"], agg["mean"], color=style["color"], linewidth=2, label=style["label"])

            has_multi = (agg["count"] > 1).any()
            if has_multi:
                y1 = agg["mean"] - agg["std"].fillna(0.0)
                y2 = agg["mean"] + agg["std"].fillna(0.0)
                ax.fill_between(agg["timestep"], y1, y2, color=style["color"], alpha=0.15)

        ax.set_title(ds_label)
        ax.set_xlabel("lead time (timestep)")
        ax.set_ylabel(metric)
        ax.grid(alpha=0.25)

    for ax in axes[n:]:
        ax.set_visible(False)

    present_groups = [g for g in family_groups_in_order(sub) if g in set(sub[group_col])]
    handles = [
        plt.Line2D([0], [0], color=styles[g]["color"], linewidth=2, label=styles[g]["label"])
        for g in present_groups
        if g in styles
    ]
    fig.legend(handles=handles, loc="upper center", ncol=max(2, len(handles)))
    plt.suptitle(f"{metric}: lead-time comparison ({PER_TIMESTEP_FILENAME})")
    plt.tight_layout(rect=(0, 0, 1, 0.94))
    plt.show()


metrics_long, missing_per_timestep = load_per_timestep_metrics(df, PER_TIMESTEP_FILENAME)

print(f"Per-timestep rows loaded: {len(metrics_long)}")
if not missing_per_timestep.empty:
    print(
        f"Missing {PER_TIMESTEP_FILENAME} for {len(missing_per_timestep)} runs "
        f"(showing first 10):"
    )
    display(missing_per_timestep.head(10))
else:
    print(f"All selected runs include {PER_TIMESTEP_FILENAME}.")

if metrics_long.empty:
    print("No per-timestep metrics found to plot.")
else:
    available_metrics = sorted(metrics_long["metric"].dropna().unique().tolist())
    print(f"Available per-timestep metrics ({len(available_metrics)}): {available_metrics}")

    preferred_metrics = ["vrmse", "rmse", "mae", "mse", "coverage_0.9", "coverage_0.5"]
    metrics_to_plot = [m for m in preferred_metrics if m in available_metrics]
    if not metrics_to_plot:
        metrics_to_plot = available_metrics[:4]

    label_order = dataset_labels_in_order(df)
    for metric_name in metrics_to_plot:
        plot_metric_vs_lead_time(
            metrics_long,
            metric=metric_name,
            dataset_labels=label_order,
            max_timestep=None,
        )
